# PointNet pour la classification de nuages de points

Ce notebook est une version découpée du script `pointnet.py`, organisée en étapes logiques avec de courtes explications.

## 1) Imports et dépendances

On charge les bibliothèques nécessaires pour le calcul, la lecture des fichiers PLY, PyTorch et les transforms.

In [6]:
!ls


sample_data


In [5]:
import numpy as np
import random
import math
import os
import time
import torch
import scipy.spatial.distance
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
import torch.nn as nn
import torch.nn.functional as F

# write_ply, read_ply sont deja importes depuis PointNetLab.Code.ply dans la cellule precedente
import sys
sys.path.append("/home/nochi/NOCHI/M2_PAR/Apprenyissage_Pointcloud/PointNetLab/Code/ply.py")
from sys import write_ply, read_ply# Performance: enable cudnn benchmark for potential speedups
torch.backends.cudnn.benchmark = True

ImportError: cannot import name 'write_ply' from 'sys' (unknown location)

## 2) Transformations et data augmentation

Ces classes appliquent des augmentations géométriques simples (rotation, bruit, permutation) puis convertissent en tenseur PyTorch.

In [ ]:
class RandomRotation_z(object):
    def __call__(self, pointcloud):
        theta = random.random() * 2. * math.pi
        rot_matrix = np.array([[math.cos(theta), -math.sin(theta),      0],
                               [math.sin(theta),  math.cos(theta),      0],
                               [0,                              0,      1]])
        rot_pointcloud = rot_matrix.dot(pointcloud.T).T
        return rot_pointcloud


class RandomNoise(object):
    def __call__(self, pointcloud):
        noise = np.random.normal(0, 0.02, (pointcloud.shape))
        noisy_pointcloud = pointcloud + noise
        return noisy_pointcloud


class ShufflePoints(object):
    def __call__(self, pointcloud):
        np.random.shuffle(pointcloud)
        return pointcloud


class ToTensor(object):
    def __call__(self, pointcloud):
        return torch.from_numpy(pointcloud)


def default_transforms():
    return transforms.Compose([
        RandomRotation_z(),
        RandomNoise(),
        ShufflePoints(),
        ToTensor()
    ])

## 3) Dataset PyTorch (ModelNet en PLY)

Cette classe indexe les fichiers `.ply`, construit les couples `(nuage, label)` et applique les transformations.

In [ ]:
class PointCloudData(Dataset):
    """
    Dataset PyTorch pour ModelNet
    """

    def __init__(self,
                 root_dir,
                 folder="train",
                 transform=default_transforms()):
        self.root_dir = root_dir

        folders = [dir for dir in sorted(os.listdir(root_dir))
                   if os.path.isdir(root_dir + "/" + dir)]

        self.classes = {folder: i for i, folder in enumerate(folders)}
        self.transforms = transform
        self.files = []

        for category in self.classes.keys():
            new_dir = root_dir + "/" + category + "/" + folder
            for file in os.listdir(new_dir):
                if file.endswith('.ply'):
                    sample = {}
                    sample['ply_path'] = new_dir + "/" + file
                    sample['category'] = category
                    self.files.append(sample)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        ply_path = self.files[idx]['ply_path']
        category = self.files[idx]['category']

        data = read_ply(ply_path)
        pointcloud = self.transforms(np.vstack((data['x'],
                                                data['y'],
                                                data['z'])).T)

        label = self.classes[category]
        return {'pointcloud': pointcloud, 'category': label}

## 4) Modèle de base MLP

Un baseline simple qui aplatit le nuage puis applique des couches fully-connected.

In [ ]:
class PointMLP(nn.Module):
    def __init__(self, classes=40):
        super().__init__()
        self.flatten = nn.Flatten(start_dim=1)

        self.fc1 = nn.Linear(3072, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.act1 = nn.ReLU()

        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.act2 = nn.ReLU()
        self.drop = nn.Dropout(0.3)

        self.fc3 = nn.Linear(256, classes)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, input):
        x = self.flatten(input)

        x = self.fc1(x)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.drop(x)

        x = self.fc3(x)
        x = self.logsoftmax(x)
        return x

## 5) PointNet basique

Version convolutionnelle 1D sur les points, suivie d'un max pooling global et d'une tête de classification.

In [ ]:
class PointNetBasic(nn.Module):
    def __init__(self, classes=40):
        super().__init__()

        self.conv1 = nn.Conv1d(3, 64, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.act1 = nn.ReLU()

        self.conv2 = nn.Conv1d(64, 64, 1)
        self.bn2 = nn.BatchNorm1d(64)
        self.act2 = nn.ReLU()

        self.conv3 = nn.Conv1d(64, 64, 1)
        self.bn3 = nn.BatchNorm1d(64)
        self.act3 = nn.ReLU()

        self.conv4 = nn.Conv1d(64, 128, 1)
        self.bn4 = nn.BatchNorm1d(128)
        self.act4 = nn.ReLU()

        self.conv5 = nn.Conv1d(128, 1024, 1)
        self.bn5 = nn.BatchNorm1d(1024)
        self.act5 = nn.ReLU()

        self.maxpool5 = nn.MaxPool1d(1024)

        self.fc6 = nn.Linear(1024, 512)
        self.bn6 = nn.BatchNorm1d(512)
        self.act6 = nn.ReLU()

        self.fc7 = nn.Linear(512, 256)
        self.bn7 = nn.BatchNorm1d(256)
        self.act7 = nn.ReLU()
        self.drop7 = nn.Dropout(0.3)

        self.fc8 = nn.Linear(256, classes)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, input):
        x = self.conv1(input)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.act3(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.act4(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.act5(x)

        x = self.maxpool5(x)
        x = x.squeeze(-1)

        x = self.fc6(x)
        x = self.bn6(x)
        x = self.act6(x)

        x = self.fc7(x)
        x = self.bn7(x)
        x = self.act7(x)
        x = self.drop7(x)

        x = self.fc8(x)
        x = self.logsoftmax(x)
        return x

## 6) T-Net et PointNet complet

Le T-Net apprend une transformation affine des points/features. Cette cellule reprend les classes du script original.

In [ ]:
class Tnet(nn.Module):
    def __init__(self, k=3):
        super().__init__()

        self.k = k

        self.conv1 = nn.Conv1d(k, 64, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.act1 = nn.ReLU()

        self.conv2 = nn.Conv1d(64, 128, 1)
        self.bn2 = nn.BatchNorm1d(128)
        self.act2 = nn.ReLU()

        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.bn3 = nn.BatchNorm1d(1024)
        self.act3 = nn.ReLU()

        self.maxpool3 = nn.MaxPool1d(1024)

        self.fc4 = nn.Linear(1024, 512)
        self.bn4 = nn.BatchNorm1d(512)
        self.act4 = nn.ReLU()

        self.fc5 = nn.Linear(512, 256)
        self.bn5 = nn.BatchNorm1d(256)
        self.act5 = nn.ReLU()

        self.fc6 = nn.Linear(256, k * k)

    def forward(self, input):
        x = self.conv1(input)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.act3(x)

        x = self.maxpool3(x)
        x = x.squeeze(-1)

        x = self.fc4(x)
        x = self.bn4(x)
        x = self.act4(x)

        x = self.fc5(x)
        x = self.bn5(x)
        x = self.act5(x)

        x = self.fc6(x)
        x = x.reshape(x.size(0), self.k, self.k)

        I = torch.eye(self.k, device=x.device, dtype=x.dtype)
        I = I.unsqueeze(0)
        I = I.repeat(x.size(0), 1, 1)
        x = x + I

        return x


class PointNetFull(nn.Module):
    def __init__(self, tnet1, tnet2, classes=40):
        super().__init__()

        self.tnet1 = tnet1
        self.tnet2 = tnet2

        self.conv1 = nn.Conv1d(3, 64, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.act1 = nn.ReLU()

        self.conv2 = nn.Conv1d(64, 64, 1)
        self.bn2 = nn.BatchNorm1d(64)
        self.act2 = nn.ReLU()

        self.conv3 = nn.Conv1d(64, 64, 1)
        self.bn3 = nn.BatchNorm1d(64)
        self.act3 = nn.ReLU()

        self.conv4 = nn.Conv1d(64, 128, 1)
        self.bn4 = nn.BatchNorm1d(128)
        self.act4 = nn.ReLU()

        self.conv5 = nn.Conv1d(128, 1024, 1)
        self.bn5 = nn.BatchNorm1d(1024)
        self.act5 = nn.ReLU()

        self.maxpool5 = nn.MaxPool1d(1024)

        self.fc6 = nn.Linear(1024, 512)
        self.bn6 = nn.BatchNorm1d(512)
        self.act6 = nn.ReLU()

        self.fc7 = nn.Linear(512, 256)
        self.bn7 = nn.BatchNorm1d(256)
        self.act7 = nn.ReLU()
        self.drop7 = nn.Dropout(0.3)

        self.fc8 = nn.Linear(256, classes)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, input):
        m3x3 = self.tnet1(input)
        x = torch.bmm(m3x3, input)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.act3(x)

        m64x64 = self.tnet2(x)
        x = torch.bmm(m64x64, x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.act4(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.act5(x)

        x = self.maxpool5(x)
        x = x.squeeze(-1)

        x = self.fc6(x)
        x = self.bn6(x)
        x = self.act6(x)

        x = self.fc7(x)
        x = self.bn7(x)
        x = self.act7(x)
        x = self.drop7(x)

        x = self.fc8(x)
        x = self.logsoftmax(x)
        return x

## 7) Fonctions de perte

`basic_loss` utilise NLLLoss. `pointnet_full_loss` ajoute une régularisation d'orthogonalité pour le T-Net.

In [ ]:
def basic_loss(outputs, labels):
    criterion = torch.nn.NLLLoss()
    return criterion(outputs, labels)


def pointnet_full_loss(outputs, labels, m3x3, alpha=0.001):
    criterion = torch.nn.NLLLoss()
    bsize = outputs.size(0)

    id3x3 = torch.eye(3, requires_grad=True).repeat(bsize, 1, 1)
    if outputs.is_cuda:
        id3x3 = id3x3.cuda()

    diff3x3 = id3x3 - torch.bmm(m3x3, m3x3.transpose(1, 2))
    return criterion(outputs, labels) + alpha * (torch.norm(diff3x3)) / float(bsize)

## 8) Boucle d'entraînement

La fonction `train` gère l'optimiseur, le scheduler, l'entraînement batch par batch et le calcul d'accuracy sur le test.

In [ ]:
def train(model, device, train_loader, test_loader=None, epochs=250):
    from torch.cuda.amp import autocast, GradScaler
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    scaler = GradScaler()

    for epoch in range(epochs):
        model.train()
        t0 = time.perf_counter()

        for i, data in enumerate(train_loader, 0):
            inputs = data['pointcloud'].to(device, non_blocking=True).float()
            labels = data['category'].to(device, non_blocking=True)

            optimizer.zero_grad()
            with autocast():
                outputs = model(inputs.transpose(1, 2))
                loss = basic_loss(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        # synchronize and report epoch time
        if device.type == 'cuda':
            torch.cuda.synchronize()
        epoch_time = time.perf_counter() - t0

        # validation
        val_acc = None
        if test_loader:
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for data in test_loader:
                    inputs = data['pointcloud'].to(device, non_blocking=True).float()
                    labels = data['category'].to(device, non_blocking=True)
                    with autocast():
                        outputs = model(inputs.transpose(1, 2))
                    _, predicted = torch.max(outputs, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
            val_acc = 100. * correct / total
            print(f'Epoch: {epoch+1}, Time: {epoch_time:.2f}s, Loss: {loss:.3f}, Test accuracy: {val_acc:.1f} %')
        else:
            print(f'Epoch: {epoch+1}, Time: {epoch_time:.2f}s, Loss: {loss:.3f}')

        scheduler.step()

## 9) Chargement du dataset et DataLoaders

On instancie les jeux d'entraînement/test puis on affiche quelques infos de contrôle.

In [ ]:
DATA_ROOT = "/home/nochi/NOCHI/M2_PAR/Apprenyissage_Pointcloud/PointNetLab/data/ModelNet10_PLY"

train_ds = PointCloudData(DATA_ROOT)
test_ds = PointCloudData(DATA_ROOT, folder='test')

inv_classes = {i: cat for cat, i in train_ds.classes.items()}
print('Classes:', inv_classes)
print('Train dataset size:', len(train_ds))
print('Test dataset size:', len(test_ds))
print('Number of classes:', len(train_ds.classes))
print('Sample pointcloud shape:', train_ds[0]['pointcloud'].size())

# DataLoader: use multiple workers and pin_memory to reduce CPU->GPU overhead
train_loader = DataLoader(dataset=train_ds, batch_size=32, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(dataset=test_ds, batch_size=32,
                         num_workers=4, pin_memory=True, persistent_workers=True)

## 10) Instanciation du modèle et configuration device

On choisit l'architecture, on compte les paramètres entraînables et on déplace le modèle sur CPU/GPU.

In [ ]:
Tnet1 = Tnet(k=3)
Tnet2 = Tnet(k=64)

# model = PointMLP(classes=10)
# model = PointNetBasic(classes=10)
# model = PointNetFull(classes=10, tnet1=Tnet1, tnet2=Tnet2)

model_parameters = filter(lambda p: p.requires_grad, model.parameters())
print('Number of parameters in the Neural Networks:', sum([np.prod(p.size()) for p in model_parameters]))

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model.to(device)

## 11) Lancement de l'entraînement

Cette cellule démarre l'entraînement complet et affiche le temps total.

In [ ]:
t0 = time.time()
train(model, device, train_loader, test_loader, epochs=20)
print('Total time for training :', time.time() - t0)